# Testing & Evaluating Agents

*Notebook 07*

An agent that works in one demo may fail on other inputs.

A structured evaluation process tests behavior across cases and catches regressions.

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool, ToolCallItem
import json
from typing import Annotated
from pydantic import BaseModel, Field

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"



print("✅ Ready!")

---

## 🤖 The Agent We'll Test

Before we can test anything, we need something to test.

We'll use a small product assistant, kept deliberately simple.

The focus stays on the eval patterns, not the agent.

In [ ]:
PRODUCTS = {
    "P001": {"name": "Wireless Headphones", "price": 149.99, "stock": 23, "rating": 4.5},
    "P002": {"name": "Phone Case", "price": 19.99, "stock": 0, "rating": 4.1},
    "P003": {"name": "Laptop Stand", "price": 49.99, "stock": 7, "rating": 4.8},
    "P004": {"name": "USB-C Cable", "price": 12.99, "stock": 150, "rating": 3.9}
}


@function_tool
def get_product(product_id: str) -> str:
    """Get details for a product by ID."""
    try:
        p = PRODUCTS[product_id.upper()]
        status = "In stock" if p["stock"] > 0 else "Out of stock"
        return (
            f"{p['name']} | ${p['price']:.2f} | "
            f"Rating: {p['rating']}/5 | {status} ({p['stock']} units)"
        )
    except KeyError:
        return f"Product '{product_id}' not found."


@function_tool
def list_products() -> str:
    """List all available products."""
    lines = [f"{pid}: {p['name']}, ${p['price']:.2f}" for pid, p in PRODUCTS.items()]
    return "\n".join(lines)

product_instructions = (
    "You are a helpful product assistant.\n"
    "Answer questions about products using the available tools.\n"
    "Always look up specific product details before answering price or stock questions.\n"
    "For comparison questions, list all products before answering.\n"
    "If a product is out of stock, mention this clearly.\n"
    "If a product is not found, say so without quoting other products' prices.\n"
    "Do not offer actions the available tools cannot perform.\n"
    "Answer the question, then stop.\n"
    "Do not ask follow-up questions or offer additional actions."
)

product_agent = Agent(
    name="ProductAssistant",
    instructions=product_instructions,
    model=MODEL,
    tools=[get_product, list_products]
)

# --------------------------------------------------------------
print("✅ Product assistant ready")
print(f"    Tools: get_product, list_products")
print(f"    Products in catalog: {len(PRODUCTS)}")

---

## ✅ Success Criteria

Decide what success looks like before writing test cases.

This product assistant is judged on three things:

- **Factual correctness**: are the prices, names, and stock statuses right?

- **Expected tool calls**: did the agent call the tools this test requires, instead of guessing?

- **Response quality**: is the answer clear and helpful?

Define your own criteria before writing a single test case.

---

## 📋 Part 1: Building a Golden Test Set

A golden test set is a set of inputs where you know the correct output.

Each test case has a question and one or more criteria the answer must satisfy.

A useful test set covers three things:

- Happy paths

- Edge cases (missing items, ambiguous inputs)

- Known failure modes you've already seen the agent get wrong

In [ ]:
# Each test case defines a question, output checks, and expected tool names.
TEST_CASES = [
    {
        "id": "T01",
        "input": "How much does product P003 cost?",
        "must_contain": ["49.99"],
        "must_not_contain": ["149.99", "19.99"],
        "must_call_tools": ["get_product"],
        "description": "Price lookup: correct product"
    },
    {
        "id": "T02",
        "input": "Is product P002 (Phone Case) in stock?",
        "must_contain": [],
        "must_contain_any": ["out of stock", "not in stock", "unavailable"],
        "must_not_contain": [],
        "must_call_tools": ["get_product"],
        "description": "Stock status: out of stock item"
    },
    {
        "id": "T03",
        "input": "What products do you carry?",
        "must_contain": [
            "Wireless Headphones",
            "Phone Case",
            "Laptop Stand",
            "USB-C Cable"
        ],
        "must_not_contain": [],
        "must_call_tools": ["list_products"],
        "description": "Product listing: all items shown"
    },
    {
        "id": "T04",
        "input": "Tell me about product P999.",
        "must_contain": ["P999"],
        "must_not_contain": ["$"],  # a graceful "not found" reply invents no price
        "must_call_tools": ["get_product"],
        "description": "Missing product: graceful error handling"
    },
    {
        "id": "T05",
        "input": "Which product has the highest rating?",
        "must_contain": ["Laptop Stand", "4.8"],
        "must_not_contain": [],
        "must_call_tools": ["list_products", "get_product"],
        "must_lookup_ids": ["P001", "P002", "P003", "P004"],
        "description": "Comparative question: list products, then look up every rating"
    },
]

# --------------------------------------------------------------
print(f"✅ Golden test set: {len(TEST_CASES)} test cases")
for tc in TEST_CASES:
    print(f"   {tc['id']}: {tc['description']}")

---

## ✅ Part 2: Pass/Fail Checks

Pass/fail checks verify two things:

- explicit strings in the response

- expected tool names, and where needed, their arguments

In [ ]:
def check_output(output: str, tool_calls: list[tuple], test_case: dict) -> tuple[bool, list[str]]:
    """Check the output text and tool calls against a test case's assertions."""
    output_lower = output.lower()
    tool_names = [name for name, _ in tool_calls]
    failures = []

    for s in test_case["must_contain"]:
        if s.lower() not in output_lower:
            failures.append(f"Missing required string: {s}")

    for s in test_case["must_not_contain"]:
        if s.lower() in output_lower:
            failures.append(f"Contains forbidden string: {s}")

    # must_contain_any: at least one acceptable phrasing must appear (e.g. synonyms)
    must_contain_any = test_case.get("must_contain_any", [])
    if must_contain_any:
        found = False
        for s in must_contain_any:
            if s.lower() in output_lower:
                found = True
        if not found:
            failures.append(f"None of the acceptable strings present: {must_contain_any}")

    # must_call_tools: the agent must actually have called these tools
    for tool in test_case.get("must_call_tools", []):
        if tool not in tool_names:
            failures.append(f"Expected tool call: {tool}")

    # must_lookup_ids: get_product must have been called for each of these IDs
    for pid in test_case.get("must_lookup_ids", []):
        looked_up = any(
            name == "get_product" and str(args.get("product_id", "")).upper() == pid
            for name, args in tool_calls
        )
        if not looked_up:
            failures.append(f"Expected get_product lookup for: {pid}")

    return not failures, failures

# --------------------------------------------------------------
print("✅ check_output() helper ready")

#### Run the Full Test Suite

In [ ]:
print("🧪 RUNNING TEST SUITE")
print("=" * 60)

results = []

for test_case in TEST_CASES:

    result = await Runner.run(product_agent, input=test_case["input"])

    output = result.final_output
    tool_calls = [
        (item.raw_item.name, json.loads(item.raw_item.arguments or "{}"))
        for item in result.new_items if isinstance(item, ToolCallItem)
    ]
    tool_names = [name for name, _ in tool_calls]

    passed, failures = check_output(output, tool_calls, test_case)

    results.append({
        "id": test_case["id"],
        "passed": passed,
        "failures": failures,
        "output": output,
        "tool_names": tool_names
    })

    icon = "✅" if passed else "❌"
    print(f"\n{icon} {test_case['id']}: {test_case['description']}")
    if not passed:
        for failure in failures:
            print(f"      → {failure}")
    print(f"   Tools: {', '.join(tool_names) if tool_names else 'none'}")
    print(f"   Output: {output}")

# Summary
passed_count = sum(1 for r in results if r["passed"])
print(f"\n{'='*60}")
print(f"📊 Results: {passed_count}/{len(results)} passed")
if passed_count == len(results):
    print("✅ All tests passed!")
else:
    print(f"❌ {len(results) - passed_count} test(s) failed. Review output above")
print("=" * 60)

### 💡 Key Takeaway

The checks are deterministic once a run returns.

They verify explicit strings and expected tool names, not subjective quality.

T05 goes one step further and checks the lookup arguments: a name check alone cannot prove every rating was fetched. How a result was used still isn't verified.

## 🏆 Part 3: Rubric-Based Evaluation with a Judge Agent

Pass/fail checks miss subjective quality like helpfulness, clarity, and tone.

A judge agent scores the response against a rubric.

The judge does not receive your data automatically, so give it:

- relevant ground truth for factual accuracy

- the agent's capability limits

This demo's catalog is small, so we pass it whole.

With a large dataset, pass only the relevant slice: for a product question, that product's entry, not the whole catalog.

In [ ]:
class EvalResult(BaseModel):
    score: Annotated[int, Field(ge=1, le=5)]
    factual_accuracy: bool
    helpful: bool
    clear: bool
    feedback: str

judge_instructions = (
    "You evaluate product assistant responses.\n\n"
    "Available actions: look up one product or list products.\n"
    "The assistant cannot place orders, send notifications, or search by category.\n\n"
    "Scoring rubric (1-5):\n"
    "5: Accurate, helpful, clear, directly answers the question\n"
    "4: Accurate and helpful, minor clarity issues\n"
    "3: Partially helpful, some inaccuracy or missing information\n"
    "2: Mostly unhelpful or inaccurate\n"
    "1: Wrong, misleading, or completely unhelpful\n\n"
    "Be strict. A score of 5 requires both correct facts AND a genuinely helpful response.\n"
    "Do not reward offers the product assistant cannot fulfill."
)

judge_agent = Agent(
    name="EvaluationJudge",
    instructions=judge_instructions,
    model=REASONING_MODEL,
    output_type=EvalResult
)

# --------------------------------------------------------------
print("✅ Judge agent ready")
print(f"    Model: {REASONING_MODEL} (judge model for this demo)")

#### Run Rubric Evaluation

In [ ]:
# Select a few test cases for rubric evaluation
eval_cases = TEST_CASES[:3]

print("🏆 RUBRIC EVALUATION")
print("=" * 60)

eval_results = []

PRODUCT_JSON = json.dumps(PRODUCTS, indent=2)
suite_results = {result["id"]: result for result in results}

for test_case in eval_cases:
    # Judge the exact response captured in the test suite above
    agent_output = suite_results[test_case["id"]]["output"]

    judge_input = (
        f"Product catalog (ground truth):\n{PRODUCT_JSON}\n\n"
        f"Question: {test_case['input']}\n\n"
        f"Agent response: {agent_output}\n\n"
        f"Evaluate this response. Use the product catalog to verify factual accuracy."
    )

    judge_result = await Runner.run(judge_agent, input=judge_input)

    evaluation = judge_result.final_output
    eval_results.append(evaluation)

    print(f"\n📝 {test_case['id']}: {test_case['description']}")
    print(f"   Score:    {evaluation.score}/5")
    print(f"   Accurate: {'✅' if evaluation.factual_accuracy else '❌'}  "
          f"Helpful: {'✅' if evaluation.helpful else '❌'}  "
          f"Clear: {'✅' if evaluation.clear else '❌'}")
    print(f"   Feedback: {evaluation.feedback}")

avg_score = sum(e.score for e in eval_results) / len(eval_results)
print(f"\n{'='*60}")
print(f"📊 Average score: {avg_score:.1f}/5")
print("=" * 60)

---

## 💪 Practice Exercise

### Exercise: Extend the Test Set

*Covers: test case design, expanding evaluation coverage*

Add three more test cases to `TEST_CASES` for scenarios not yet covered, then run the full suite.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Extend the Test Set
# --------------------------------------------------------------
# Objective: Add edge cases the current test set doesn't cover.

# TODO 1: Think of 3 scenarios the current tests don't cover.
#          Ideas: asking about price of cheapest item, asking in a different
#          phrasing, asking about a real product by name instead of ID,
#          multi-part questions, ambiguous queries.

# TODO 2: Add 3 new test cases to TEST_CASES following the same format:
#          {"id": ..., "input": ..., "must_contain": [...],
#           "must_not_contain": [...], "must_call_tools": [...],
#           "description": ...}

# TODO 3: Re-run the test suite on all cases (original + new)
#          Print a summary showing which tests pass and which fail

# Optional stretch: judge one new response and compare its score
# with the deterministic pass/fail result.

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Build a golden test set early:**

- Include happy paths, edge cases, and known failure modes

- Each test case needs an input and verifiable expected facts

- Aim for 10-20 high-signal cases

- Add a case whenever a real failure appears
<br>
<br>

**Pass/fail checks catch two kinds of failure:**

- `must_contain` and `must_not_contain` check explicit strings

- `must_call_tools` catches right answers produced by guessing instead of using the required tool

- Once a run returns, these checks are fast, deterministic, and easy to extend
<br>
<br>

**Judge agents score subjective quality:**

- This demo's judge uses `REASONING_MODEL` with structured output; validate judge reliability on labeled cases before picking a model

- Pass ground truth and capability limits into the judge prompt

- Define a clear rubric to reduce scoring drift

- Numeric scores make results easier to compare across runs

- Judge scores are signals, not ground truth

---

## 📍 Next Step

**Notebook 08: Web Search**  

Give your agent access to live information from the web using OpenAI's built-in web search tool.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-07-testing-and-evaluating-agents)

---